# Train and evaluate the user-sequence model

Reads `Features/v2`, trains one model per fold of the training users (5 folds, stratified by label), picks the
decision rule (probability threshold and number of consecutive posts) from the pooled out-of-fold predictions, and
evaluates the ensemble of the fold models once on the test users. Finished experiments are cached in
`models/v2/classifier/<name>/`, so re-running a cell does not train again unless the settings changed.

Run with the `hongyi` kernel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import experiment as ex
import seq_eval as se
import seq_model as sm
from feature_cols import EXPERIMENTS_V2
from paths import Paths

P = Paths()
DEVICE = sm.pick_device()
train_df, test_df = ex.load_features(P)
if P.embeddings("train").exists():  # frozen MentalRoBERTa embeddings from embed_posts.py, for the mental_* experiments
    train_df, test_df = ex.attach_embeddings(P, train_df, "train"), ex.attach_embeddings(P, test_df, "test")
print(DEVICE)
for name, d in (("train", train_df), ("test", test_df)):
    users = d.groupby("user_id")["Depression"].max()
    print(f"{name}: users={len(users)} (dep={int(users.sum())}) posts={len(d)}")

## Settings
`all` is the full model (148 features); the other names are ablations: `only_<group>` keeps one feature group (plus the
context features), `without_<group>` drops one group, `post_level_only` uses no history features, `base_features` is the
earlier 101-column set and `no_cummean` drops the running means. Groups are PHQ, SIM, LSM, RE, OWN_LIWC, BEHAVIOR.

In [ ]:
cfg = ex.Config()  # 5 folds, seed 42, up to 30 epochs with early stopping; see experiment.Config
RUN = ["all", "lean", "lean2", "base_features", "no_cummean", "post_level_only",
       "only_PHQ", "only_OWN_LIWC", "only_BEHAVIOR", "only_RE", "only_LSM", "only_SIM",
       "without_PHQ", "without_SIM", "without_LSM", "without_RE", "without_OWN_LIWC", "without_BEHAVIOR",
       "lean_PHQ_plus_RE", "lean_PHQ_plus_LIWC", "lean_PHQ_plus_LSM",
       "lean_without_PHQ", "lean_without_RE", "lean_without_LIWC", "lean_without_LSM",
       "lean_without_cummean", "lean_without_post_index",
       "only_LSMX", "lean_PHQ_plus_LSMX", "lean_plus_LSMX", "all_v3",
       "only_LSMX_no_par", "only_LSMX_par", "lean_PHQ_plus_LSMX_no_par", "lean_PHQ_plus_LSMX_par"]
RUN += [f"lean2_without_{p}" for p in ("PHQ", "RE", "LIWC", "LSMalign", "lsmr", "dlsm", "cummean", "post_index")]
RUN += ["mental_emb", "mental_plus_lean2"]  # frozen MentalRoBERTa embeddings, same sequence model
RUN += ["ft_logit", "ft_logit_plus_lean2"]  # adapter-fine-tuned MentalRoBERTa logit, same sequence model
print({name: len(EXPERIMENTS_V2[name]) for name in RUN})

## Train and evaluate
One model per fold for every experiment in `RUN`; the test users are scored only after the decision rule is fixed.

In [ ]:
results = {name: ex.run_experiment(name, EXPERIMENTS_V2[name], train_df, test_df, P, cfg, device=DEVICE) for name in RUN}

## Comparison
`oof_*` are the out-of-fold scores on the training users at the chosen rule; `test_*` come from the fold ensemble on
the test users (`test_f1_ci` is a 95% bootstrap interval over users). `auc@n` is the user-level AUC of the probability
after the first n posts, which does not reward simply having more posts.

In [ ]:
ex.summary_table(results)

### Spread over random seeds
Training is not deterministic on the GPU, and each seed also re-draws the folds, so a single run can move the test F1
by a few hundredths. Train the other seeds from the command line (they are cached like everything else):
`python run_experiments.py all only_PHQ ... --seeds 42 43 44`.

In [ ]:
SEEDS = [42, 43, 44]
ex.seed_summary(P, RUN, SEEDS)

## The full model in detail

In [ ]:
r = results["all"]
op = r["operating_point"]
print(f"decision rule: flag after k={op['k']} consecutive posts with probability >= {op['th']}")
print("out-of-fold:", {m: round(v, 3) for m, v in op["oof"].items() if v is not None})
t = r["test"]
for m in ("precision", "recall", "f1", "median_delay", "f_latency"):
    lo, hi = t["ci"][m]
    print(f"test {m:13s} {t[m]:.3f}  (95% CI {lo:.3f} - {hi:.3f})")
print("test AUC of the probability after the first n posts:", t["horizon_auc"])
print("test F1 of each fold model:", [round(f["f1"], 3) for f in t["per_fold"]])

In [ ]:
probs = ex.load_probs(P, "all")["test"]
labels, curves, th = probs["labels"], probs["probs"], op["th"]
delay = se.delays(curves, th, [op["k"]])[:, 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(delay[(labels == 1) & (delay > 0)], bins=30)
axes[0].set(title="Posts read before the alarm (depressed test users)", xlabel="posts", ylabel="users")
rng = np.random.default_rng(0)
shown = list(rng.choice(np.flatnonzero(labels == 1), 2, replace=False)) + list(rng.choice(np.flatnonzero(labels == 0), 2, replace=False))
for i in shown:
    axes[1].plot(curves[i], label=f"{'dep' if labels[i] else 'ctrl'} {probs['users'][i]}")
axes[1].axhline(th, color="red", linestyle="--")
axes[1].set(title="Probability after each post", xlabel="post", ylabel="probability")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

## Errors on the test users
Depressed users that were never flagged and control users that were flagged, with the highest probability each reached.

In [ ]:
per_user = pd.DataFrame({"user": probs["users"], "label": labels, "posts": [len(c) for c in curves],
                         "delay": delay, "peak": [c.max() for c in curves]})
wrong = ((per_user.label == 1) & (per_user.delay == 0)) | ((per_user.label == 0) & (per_user.delay > 0))
print("missed depressed users:", int(((per_user.label == 1) & (per_user.delay == 0)).sum()),
      "| flagged control users:", int(((per_user.label == 0) & (per_user.delay > 0)).sum()))
per_user[wrong].sort_values("peak", ascending=False)